In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -----------------------------
# Step 1: Load merged dataset
# -----------------------------
file_path = r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\merged_dataset.csv"
df = pd.read_csv(file_path)
print("Original dataset size:", len(df))

# -----------------------------
# Step 2: Sample 10k rows
# -----------------------------
df_subset = df.sample(n=10000, random_state=42).reset_index(drop=True)
subset_file = r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\merged_dataset_10k.csv"
df_subset.to_csv(subset_file, index=False)
print("✅ 10k dataset saved to:", subset_file)

# -----------------------------
# Step 3: Split into train/val/test
# -----------------------------
train_df, temp_df = train_test_split(df_subset, test_size=0.3, random_state=42, stratify=df_subset["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

# -----------------------------
# Step 4: Save the splits
# -----------------------------
train_df.to_csv(r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\train_10k.csv", index=False)
val_df.to_csv(r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\val_10k.csv", index=False)
test_df.to_csv(r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\test_10k.csv", index=False)

print(f"✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


Original dataset size: 724682
✅ 10k dataset saved to: C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\merged_dataset_10k.csv
✅ Train: 7000 | Val: 1500 | Test: 1500


In [2]:
import pandas as pd

train_df = pd.read_csv(r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\train_10k.csv")
val_df   = pd.read_csv(r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\val_10k.csv")
test_df  = pd.read_csv(r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\test_10k.csv")
print("done")

done


In [4]:
# -----------------------------
# Imports
# -----------------------------
import torch
import os
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# -----------------------------
# Paths
# -----------------------------
save_dir = r"C:\Users\sonu\Desktop\big_data\project\ML\bert"
os.makedirs(save_dir, exist_ok=True)
print("fetched path")
train_path = r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\train_10k.csv"
val_path   = r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\val_10k.csv"
test_path  = r"C:\Users\sonu\Desktop\big_data\project\Original Reddit Data\test_10k.csv"
print("reading")

# -----------------------------
# Load splits
# -----------------------------
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print(f"✅ Loaded splits: {len(train_df)} train | {len(val_df)} val | {len(test_df)} test")

# -----------------------------
# Encode labels
# -----------------------------
label_encoder = LabelEncoder()
train_df["label_id"] = label_encoder.fit_transform(train_df["label"])
val_df["label_id"]   = label_encoder.transform(val_df["label"])
test_df["label_id"]  = label_encoder.transform(test_df["label"])
num_labels = len(label_encoder.classes_)

print("✅ Labels encoded:", dict(zip(label_encoder.classes_, range(num_labels))))

# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def batch_tokenize(texts, batch_size=500, max_length=128):
    input_ids, attention_masks = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")
        input_ids.append(enc["input_ids"])
        attention_masks.append(enc["attention_mask"])
    return {
        "input_ids": torch.cat(input_ids, dim=0),
        "attention_mask": torch.cat(attention_masks, dim=0)
    }

# -----------------------------
# Save/load tokenized encodings
# -----------------------------
train_enc_file = os.path.join(save_dir, "train_encodings.pt")
val_enc_file   = os.path.join(save_dir, "val_encodings.pt")
test_enc_file  = os.path.join(save_dir, "test_encodings.pt")

if os.path.exists(train_enc_file):
    train_encodings = torch.load(train_enc_file)
    val_encodings   = torch.load(val_enc_file)
    test_encodings  = torch.load(test_enc_file)
    print("✅ Loaded tokenized encodings from disk")
else:
    train_encodings = batch_tokenize(train_df["text"].tolist())
    val_encodings   = batch_tokenize(val_df["text"].tolist())
    test_encodings  = batch_tokenize(test_df["text"].tolist())
    torch.save(train_encodings, train_enc_file)
    torch.save(val_encodings, val_enc_file)
    torch.save(test_encodings, test_enc_file)
    print("✅ Tokenized encodings saved to disk")

# -----------------------------
# PyTorch Dataset
# -----------------------------
class RedditDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = RedditDataset(train_encodings, train_df["label_id"].tolist())
val_dataset   = RedditDataset(val_encodings, val_df["label_id"].tolist())
test_dataset  = RedditDataset(test_encodings, test_df["label_id"].tolist())

print("✅ Datasets ready!")

# -----------------------------
# Load DistilBERT
# -----------------------------
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)
print(f"✅ Model loaded for {num_labels} labels")

# Save model and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Model and tokenizer saved to {save_dir}")

# -----------------------------
# Metrics
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# -----------------------------
# Training arguments (CPU-friendly)
# -----------------------------
training_args = TrainingArguments(
    output_dir=os.path.join(save_dir, "results"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir=os.path.join(save_dir, "logs"),
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    eval_steps=200,
    fp16=False,
    no_cuda=True
)

# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -----------------------------
# Train
# -----------------------------
trainer.train()
print("✅ Training complete")

# -----------------------------
# Evaluate on test set
# -----------------------------
results = trainer.evaluate(test_dataset)
print("✅ Test Results:", results)


fetched path
reading
✅ Loaded splits: 7000 train | 1500 val | 1500 test
✅ Labels encoded: {'anx': 0, 'dep': 1, 'lone': 2, 'mh': 3, 'sw': 4}


C:\Users\sonu\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7699bd8d-abc0-49a8-b183-e3c510343e9f)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt
Retrying in 1s [Retry 1/5].
Tokenizing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:25<00:00,  8.54s/it]


✅ Tokenized encodings saved to disk
✅ Datasets ready!


Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.

✅ Model loaded for 5 labels
✅ Model and tokenizer saved to C:\Users\sonu\Desktop\big_data\project\ML\bert


C:\Users\sonu\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 7000
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 1314
  Number of trainable parameters = 66957317


Step,Training Loss



KeyboardInterrupt



In [7]:
import torch
import transformers
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("cuDNN version:", torch.backends.cudnn.version())
print("Transformers version:", transformers.__version__)


Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch version: 2.2.0+cpu
CUDA available: False
Transformers version: 4.25.1
